# Agent Evaluation Pipeline

This notebook runs the evaluation pipeline for comparing human survey answers with AI-generated answers.

The pipeline has three evaluation layers:

1. Semantic similarity
2. Gemini-as-judge scoring
3. Content-level comparison

Outputs are saved in `outputs/`.


## Setup

Install the project dependencies from `requirements.txt` before running the notebook.

Example:

```bash
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for path in [current, *current.parents]:
        has_src = (path / "src").exists()
        has_data = (path / "data").exists()
        has_requirements = (path / "requirements.txt").exists()

        if has_src and has_data and has_requirements:
            return path

    raise FileNotFoundError("Project root not found. Run the notebook from inside the repo.")


PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"
DATA_PATH = PROJECT_ROOT / "data" / "RB_GenAI_Datatest.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("Data file:", DATA_PATH)
print("Output folder:", OUTPUT_DIR)
print("Data exists:", DATA_PATH.exists())

## Run settings

In [ ]:
EMBEDDING_MODEL = "Alibaba-NLP/gte-large-en-v1.5"
JUDGE_MODEL = "gemini-2.5-flash"
RATE_LIMIT_SECONDS = 20

if not os.environ.get("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY is missing. Add it to your environment or .env file.")

print("Embedding model:", EMBEDDING_MODEL)
print("Judge model:", JUDGE_MODEL)
print("Pause between LLM calls:", RATE_LIMIT_SECONDS, "seconds")

## Load input data

In [ ]:
from agent_eval.io_utils import load_input_data

df = load_input_data(DATA_PATH)

print("Rows, columns:", df.shape)
df.head()

## Semantic similarity

This step measures broad meaning overlap between the human answer and the AI-generated answer.


In [ ]:
from agent_eval.semantic_similarity import run_semantic_similarity

SEMANTIC_PATH = OUTPUT_DIR / "semantic_similarity_results.xlsx"

semantic_df = run_semantic_similarity(
    input_path=DATA_PATH,
    output_path=SEMANTIC_PATH,
    model_name=EMBEDDING_MODEL,
    batch_size=16,
)

print("Saved:", SEMANTIC_PATH)
semantic_df[
    [
        "id",
        "question_category",
        "human_answers",
        "ai_answers",
        "semantic_similarity_score",
    ]
].head()

In [ ]:
semantic_df["semantic_similarity_score"].describe()

## Gemini-as-judge evaluation

This step checks how well the AI answer simulates the specific human answer across behavior, preference, reasoning, detail preservation, groundedness, style, and overall match.


In [ ]:
from agent_eval.llm_as_judge import run_llm_judge

JUDGE_PATH = OUTPUT_DIR / "llm_judge_results.xlsx"

judge_df = run_llm_judge(
    input_path=SEMANTIC_PATH,
    output_path=JUDGE_PATH,
    model_name=JUDGE_MODEL,
    sleep_seconds=RATE_LIMIT_SECONDS,
)

print("Saved:", JUDGE_PATH)
judge_df[
    [
        "id",
        "semantic_similarity_score",
        "behavior_match_score",
        "preference_match_score",
        "reasoning_match_score",
        "detail_preservation_score",
        "unsupported_additions_control_score",
        "response_style_match_score",
        "overall_simulation_match_score",
        "final_verdict",
        "main_failure_type",
    ]
].head()

## Content-level comparison

This step breaks each answer into small content elements and compares what was preserved, omitted, changed, or unsupported.


In [ ]:
from agent_eval.content_comparison import run_content_comparison

CONTENT_PATH = OUTPUT_DIR / "content_comparison_results.xlsx"

content_df = run_content_comparison(
    input_path=JUDGE_PATH,
    output_path=CONTENT_PATH,
    model_name=JUDGE_MODEL,
    sleep_seconds=RATE_LIMIT_SECONDS,
)

print("Saved:", CONTENT_PATH)
content_df[
    [
        "id",
        "content_recall",
        "content_precision",
        "content_f1",
        "unsupported_addition_rate",
        "omission_rate",
        "content_comparison_summary",
    ]
].head()

## Final analysis outputs

In [ ]:
from agent_eval.analysis import save_analysis_outputs

final_df = pd.read_excel(CONTENT_PATH)

tables = save_analysis_outputs(
    df=final_df,
    output_dir=OUTPUT_DIR,
)

print("Saved final outputs")
tables["metric_summary"]

## Category summary

In [ ]:
tables["category_summary"]

## Failure-type summary

In [ ]:
tables["failure_type_summary"]

## Verdict summary

In [ ]:
tables["verdict_summary"]

## High semantic similarity, weak alignment

These rows are useful for explaining why semantic similarity alone is not enough.


In [ ]:
tables["high_semantic_low_alignment"]

## Best and worst examples

In [ ]:
tables["best_examples"]

In [ ]:
tables["worst_examples"]

## Output files

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)

In [ ]:
FINAL_RESULTS = OUTPUT_DIR / "final_combined_results.xlsx"
ANALYSIS_SUMMARY = OUTPUT_DIR / "analysis_summary.xlsx"

print("Final results:", FINAL_RESULTS)
print("Analysis summary:", ANALYSIS_SUMMARY)